In [4]:
import pandas as pd
import numpy as np


df_lucro_ano = pd.read_csv("../data/processed/lucro_liquido_2021_2025.csv")
df_pl_ano = pd.read_csv("../data/processed/patrimonio_liquido_2021_2025.csv")

df_roe = df_lucro_ano.merge(df_pl_ano, on=["DENOM_CIA", "ANO_ARQUIVO"], suffixes=("_LUCRO", "_PL"))
df_roe["ROE"] = df_roe["VL_CONTA_LUCRO"] / df_roe["VL_CONTA_PL"]

df_roe.sort_values("ROE", ascending=False).head(10)

,DENOM_CIA,SETOR_ATIV,ANO_ARQUIVO,VL_CONTA_LUCRO,VL_CONTA_REAIS,VL_CONTA_BI,VL_CONTA_PL,ROE
1785,CLI SUL S.A.,Serviços Transporte e Logística,2024,120746.0,1.207460e+08,0.120746,0.0,inf
1655,RIO PARANAPANEMA ENERGIA S.A.,Energia Elétrica,2024,683234.0,6.832340e+08,0.683234,0.0,inf
1803,TIM S.A.,Telecomunicações,2024,2837422.0,2.837422e+09,2.837422,0.0,inf
1200,RIO PARANAPANEMA ENERGIA S.A.,Energia Elétrica,2023,1169684.0,1.169684e+09,1.169684,0.0,inf
187,BRAZILIAN FINANCE E REAL ESTATE S.A.,Emp. Adm. Part. - Crédito Imobiliário,2021,20748.0,2.074800e+07,0.020748,0.0,inf
2226,CLI SUL S.A.,Serviços Transporte e Logística,2025,57107.0,5.710700e+07,0.057107,0.0,inf
2131,B100 S.A.,Intermediação Financeira,2025,-465085.0,-4.650850e+08,-0.465085,-746.0,623.438338
682,MANGELS INDUSTRIAL S.A.,Emp. Adm. Part. - Metalurgia e Siderurgia,2022,408054.0,4.080540e+08,0.408054,7225.0,56.478062
1821,KARSTEN S.A.,Têxtil e Vestuário,2024,160569.0,1.605690e+08,0.160569,5469.0,29.359846
2026,TERP GLBL BRASIL I PARTICIPAÇÕES S.A.,Emp. Adm. Part. - Energia Elétrica,2025,215452.0,2.154520e+08,0.215452,7938.0,27.141849


In [ ]:
df_roe[df_roe["VL_CONTA_PL"] == 0][["DENOM_CIA", "ANO_ARQUIVO", "VL_CONTA_LUCRO", "VL_CONTA_PL"]]
#Aqui deu um probleminha porque algumas empresas não tinham patrimônio líquido, então o ROE deu infinito. Então vamos filtrar essas empresas para ver o que aconteceu.

,DENOM_CIA,ANO_ARQUIVO,VL_CONTA_LUCRO,VL_CONTA_PL
187,BRAZILIAN FINANCE E REAL ESTATE S.A.,2021,20748.0,0.0
509,CLARANET TECHNOLOGY S.A.,2022,-12180.0,0.0
666,CIA CELG DE PARTICIPACOES - CELGPAR,2022,0.0,0.0
1119,CIA CELG DE PARTICIPACOES - CELGPAR,2023,0.0,0.0
1200,RIO PARANAPANEMA ENERGIA S.A.,2023,1169684.0,0.0
1655,RIO PARANAPANEMA ENERGIA S.A.,2024,683234.0,0.0
1785,CLI SUL S.A.,2024,120746.0,0.0
1803,TIM S.A.,2024,2837422.0,0.0
2108,RIO PARANAPANEMA ENERGIA S.A.,2025,0.0,0.0
2226,CLI SUL S.A.,2025,57107.0,0.0


In [7]:
# remove linhas com PL zerado (dado incompleto/inconsistente da fonte)
antes = len(df_roe)
df_roe = df_roe[df_roe["VL_CONTA_PL"] != 0].copy()
depois = len(df_roe)
print(f"Removidas {antes - depois} linhas com PL zerado (dado inconsistente da CVM)")


Removidas 11 linhas com PL zerado (dado inconsistente da CVM)


In [8]:
df_roe[df_roe["ROE"] < 0][["DENOM_CIA", "ANO_ARQUIVO", "VL_CONTA_LUCRO", "VL_CONTA_PL", "ROE"]].sort_values("ROE")

,DENOM_CIA,ANO_ARQUIVO,VL_CONTA_LUCRO,VL_CONTA_PL,ROE
1008,MARISA LOJAS S.A.,2023,-899241.0,4.670000e+02,-1925.569593
148,ALPHAVILLE S.A.,2021,-801463.0,1.279600e+04,-62.633870
1266,RECRUSUL S.A.,2023,-17873.0,4.730000e+02,-37.786469
184,NEXPE PARTICIPAÇÕES S.A,2021,-251030.0,9.856000e+03,-25.469765
1049,VIVER INCORPORADORA E CONSTRUTORA S.A.,2023,-105988.0,5.293000e+03,-20.024183
...,...,...,...,...,...
596,GAFISA S.A.,2022,-2142.0,1.772906e+06,-0.001208
1666,AXIA ENERGIA SUL S.A.,2024,-8085.0,8.805726e+06,-0.000918
2029,FOZ DO RIO CLARO ENERGIA S.A.,2025,-213.0,3.550270e+05,-0.000600
92,TECHNOS S.A.,2021,-82.0,3.290140e+05,-0.000249


In [9]:
LIMITE_PL_MINIMO = 10_000_000  # ajuste esse valor conforme necessário

df_roe_filtrado = df_roe[df_roe["VL_CONTA_PL"].abs() >= LIMITE_PL_MINIMO].copy()

print(f"Removidas {len(df_roe) - len(df_roe_filtrado)} linhas com PL abaixo do limite de materialidade")
df_roe_filtrado.sort_values("ROE", ascending=False).head(10)

Removidas 1869 linhas com PL abaixo do limite de materialidade


,DENOM_CIA,SETOR_ATIV,ANO_ARQUIVO,VL_CONTA_LUCRO,VL_CONTA_REAIS,VL_CONTA_BI,VL_CONTA_PL,ROE
766,VALE S.A.,Extração Mineral,2022,435360000.0,4.353600e+11,435.360000,194894000.0,2.233830
1638,ENGIE BRASIL ENERGIA S.A.,Energia Elétrica,2024,23195703.0,2.319570e+10,23.195703,12280398.0,1.888840
2052,BB SEGURIDADE PARTICIPAÇÕES S.A.,Emp. Adm. Part. - Seguradoras e Corretoras,2025,17720682.0,1.772068e+10,17.720682,10384393.0,1.706473
1282,PETROLEO BRASILEIRO S.A. PETROBRAS,Petróleo e Gás,2023,628342000.0,6.283420e+11,628.342000,382340000.0,1.643412
827,PETROLEO BRASILEIRO S.A. PETROBRAS,Petróleo e Gás,2022,592538000.0,5.925380e+11,592.538000,364385000.0,1.626132
2086,ENGIE BRASIL ENERGIA S.A.,Energia Elétrica,2025,21482280.0,2.148228e+10,21.482280,13914493.0,1.543878
326,CIA SIDERURGICA NACIONAL,Metalurgia e Siderurgia,2021,35776478.0,3.577648e+10,35.776478,23374389.0,1.530585
307,VALE S.A.,Extração Mineral,2021,292492000.0,2.924920e+11,292.492000,197058000.0,1.484294
784,CIA SIDERURGICA NACIONAL,Metalurgia e Siderurgia,2022,31526638.0,3.152664e+10,31.526638,21816044.0,1.445113
556,AMERICANAS S.A. - EM RECUPERAÇÃO JUDICIAL,Comércio (Atacado e Varejo),2022,-38297800.0,-3.829780e+10,-38.297800,-26666621.0,1.436170


In [11]:
#Antes de importar, vamos mostrar uma coluna nova mostrando quando VL_CONTA_PL é negativo, para ver se tem alguma empresa com patrimônio líquido negativo. Se tiver, o ROE é enganoso.
df_roe_filtrado["ROE_ENGANOSO"] = df_roe_filtrado["VL_CONTA_PL"] < 0
df_roe_filtrado.to_csv("../data/processed/roe_2021_2025.csv", index=False)